# Tabletop detection - controlled experiments on Kaggle (GPU)

Trains several YOLO11 variants on the same data, **changing one thing at a time**, with several seeds each,
and compares them on the **validation** split (session s05).

- train = sessions s01-s04, val = session s05.
- The test split (s06) is **not** in this notebook on purpose. It is evaluated once, locally, at the very end.

Before running: enable **Internet** and a **GPU** (T4 / P100) in the notebook settings, and add the private dataset
`tabletop-dataset` (the `tabletop_dataset.zip` archive) as input.

In [ ]:
!pip install -q ultralytics==8.4.174
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU in the settings")

In [ ]:
import shutil
from pathlib import Path
import yaml

# Locate the uploaded dataset (read-only input) and work on a writable copy.
candidates = list(Path("/kaggle/input").glob("**/dataset/train/images"))
assert candidates, "Dataset not found under /kaggle/input: add tabletop-dataset as notebook input."
src = candidates[0].parent.parent
dst = Path("/kaggle/working/dataset")
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)

NAMES = ["airpod", "charger", "fork", "pen", "spoon", "usb_key"]
DATA = "/kaggle/working/dataset.yaml"
Path(DATA).write_text(yaml.safe_dump({"path": str(dst), "train": "train/images", "val": "val/images",
                                       "names": NAMES}, sort_keys=False))
for split in ("train", "val"):
    print(split, len(list((dst / split / "images").glob("*.jpg"))), "images")
assert not (dst / "test").exists(), "the test split must not be in this notebook"

## Experiments

Run 1 on the Mac (MPS) used `yolo11n.pt`, 30 epochs, imgsz 640, batch 16, patience 10. `baseline_yolo11n` reproduces it here.
Every other entry changes one factor relative to that baseline, except `scratch_yolo11n`, which gets 100 epochs because
random initialisation needs a bigger budget to be a fair reference.

In [ ]:
SEEDS = [0, 1, 2]   # each experiment is repeated: differences of 1-2 points are noise with only 80 val images

EXPERIMENTS = {
    "baseline_yolo11n": dict(model="yolo11n.pt", epochs=30),
    "scratch_yolo11n":  dict(model="yolo11n.yaml", pretrained=False, epochs=100, patience=30),
    "aug_strong":       dict(model="yolo11n.pt", epochs=30, hsv_s=0.9, hsv_v=0.6, degrees=10, translate=0.2, mixup=0.1),
    "yolo11s":          dict(model="yolo11s.pt", epochs=30),
    "epochs60":         dict(model="yolo11n.pt", epochs=60, patience=20),
    "imgsz960":         dict(model="yolo11n.pt", epochs=30, imgsz=960),
}

COMMON = dict(data=DATA, imgsz=640, batch=16, patience=10, device=0, workers=2,
              plots=False, verbose=False, exist_ok=True, project="/kaggle/working/runs")

In [ ]:
import time
from ultralytics import YOLO

rows = []
for exp, cfg in EXPERIMENTS.items():
    for seed in SEEDS:
        t0 = time.time()
        args = {**COMMON, **cfg, "seed": seed, "name": f"{exp}_s{seed}"}
        model = YOLO(args.pop("model"))
        model.train(**args)

        best = YOLO(f"/kaggle/working/runs/{args['name']}/weights/best.pt")
        m = best.val(data=DATA, split="val", imgsz=args["imgsz"], device=0, plots=False, verbose=False)
        row = {"experiment": exp, "seed": seed, "mAP50": m.box.map50, "mAP50-95": m.box.map,
               "P": m.box.mp, "R": m.box.mr, "minutes": (time.time() - t0) / 60}
        for i, c in enumerate(m.box.ap_class_index):
            row[f"AP50-95_{NAMES[int(c)]}"] = m.box.ap[i]
            row[f"AP50_{NAMES[int(c)]}"] = m.box.ap50[i]
        rows.append(row)
        print(f"{exp} seed={seed}: mAP50={row['mAP50']:.3f} mAP50-95={row['mAP50-95']:.3f} ({row['minutes']:.1f} min)")

In [ ]:
import pandas as pd
df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/all_runs.csv", index=False)

cols = ["mAP50", "mAP50-95", "P", "R"] + [f"AP50-95_{n}" for n in NAMES]
summary = df.groupby("experiment", sort=False)[cols].agg(["mean", "std"]).round(3)
summary.to_csv("/kaggle/working/summary_mean_std.csv")
pd.set_option("display.width", 220)
print(summary[["mAP50", "mAP50-95"]])
summary

In [ ]:
# Package results: metrics + per-run curves, and the best weights of seed 0 only (small files).
import zipfile
with zipfile.ZipFile("/kaggle/working/results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["all_runs.csv", "summary_mean_std.csv"]:
        z.write(f"/kaggle/working/{f}", f)
    for p in Path("/kaggle/working/runs").glob("*/results.csv"):
        z.write(p, f"runs/{p.parent.name}/results.csv")
    for p in Path("/kaggle/working/runs").glob("*_s0/weights/best.pt"):
        z.write(p, f"weights/{p.parent.parent.name}_best.pt")
print("results.zip ready in /kaggle/working (download it from the Output tab)")